# Consolidated official gaming revenue

This notebook **reads SQLite** and rebuilds the processed CSVs.
It contains **no downloading or parsing**.

GGR, AGR, taxable revenue, and net proceeds stay labeled. Do not sum them as if they were one metric.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.consolidate import export_all
from variant_gaming.storage import connect, default_db_path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else ROOT
from variant_gaming.common import project_root
ROOT = project_root()
paths = export_all(ROOT)
paths

{'gaming_results': WindowsPath('C:/Users/Sean/VscProjects/researchOS/data/processed/gaming_results.csv'),
 'state_period_revenue': WindowsPath('C:/Users/Sean/VscProjects/researchOS/data/processed/state_period_revenue.csv'),
 'operator_revenue': WindowsPath('C:/Users/Sean/VscProjects/researchOS/data/processed/operator_revenue.csv'),
 'source_coverage': WindowsPath('C:/Users/Sean/VscProjects/researchOS/data/processed/source_coverage.csv')}

## Coverage first

In [2]:
coverage = pd.read_csv(paths["source_coverage"])
print(coverage.groupby("status").size())
coverage.sort_values(["status", "state_code"])

status
annual_only                       1
blocked                           3
blocked_or_unavailable_export     1
combined_only                     2
legal_not_reporting               1
location_based_mobile             1
not_publicly_available            2
ok                               20
on_premises_only                  1
partial                           3
pdf_only_not_yet_parsed           7
dtype: int64


,state_code,vertical,status,reason,official_url,available_frequency,earliest_period,latest_period,downloaded_file_count,normalized_row_count,last_retrieval_utc
31,OR,online_sports_betting,annual_only,Official public reporting is mainly annual/sum...,https://www.oregonlottery.org/annual-report-2025/,annual,NaN,NaN,0,0,2026-09-03T01:49:14.251096+00:00
1,AZ,online_sports_betting,blocked,Official reports landing blocked for automated...,https://gaming.az.gov/resources/reports,monthly,NaN,NaN,0,0,2026-09-03T01:48:51.681257+00:00
7,DE,online_sports_betting,blocked,Official tables split casino sportsbooks vs Sp...,https://delottery.com/Sports-Lottery/Monthly-N...,monthly,NaN,NaN,20,0,2026-09-03T01:43:25.677259+00:00
13,KY,online_sports_betting,blocked,Current sports wagering market report is Table...,https://khrc.ky.gov/new_docs.aspx?cat=76&menui...,monthly,NaN,NaN,0,0,2026-09-03T01:48:52.602236+00:00
41,WY,online_sports_betting,blocked_or_unavailable_export,Combined wagering activity landing returned no...,https://gaming.wyo.gov/revenue-reports/financi...,monthly,NaN,NaN,0,0,2026-09-03T01:49:11.358670+00:00
0,AR,online_sports_betting,combined_only,Official public landing does not expose an iso...,https://www.dfa.arkansas.gov/office/taxes/exci...,NaN,NaN,NaN,0,0,2026-09-03T01:49:12.747031+00:00
28,NV,online_sports_betting,combined_only,Official GRI monthly PDFs provide sports-pool ...,https://www.gaming.nv.gov/about-us/gaming-reve...,monthly,NaN,NaN,0,0,2026-09-03T01:49:13.469229+00:00
17,ME,online_casino,legal_not_reporting,"I-Gaming authorized in law, but no official op...",https://www.maine.gov/dps/gcu/I-Gaming,NaN,NaN,NaN,0,0,2026-09-03T01:49:12.761313+00:00
23,MT,location_based_mobile_sports_betting,location_based_mobile,Montana sports wagers are tied to sales-agent ...,https://montanalottery.com/,weekly,NaN,NaN,0,0,2026-09-03T01:49:13.463322+00:00
8,FL,online_sports_betting,not_publicly_available,No official monthly online sportsbook series o...,https://flgaming.gov/pmw/statistics/,NaN,NaN,NaN,0,0,2026-09-03T01:49:12.753497+00:00


## Missing, blocked, and special-case sources

In [3]:
watch = coverage[~coverage["status"].isin(["ok"])]
watch[["state_code", "vertical", "status", "reason", "official_url"]]

,state_code,vertical,status,reason,official_url
0,AR,online_sports_betting,combined_only,Official public landing does not expose an iso...,https://www.dfa.arkansas.gov/office/taxes/exci...
1,AZ,online_sports_betting,blocked,Official reports landing blocked for automated...,https://gaming.az.gov/resources/reports
2,CO,online_sports_betting,pdf_only_not_yet_parsed,Official monthly Sports Betting Proceeds PDFs ...,https://sbg.colorado.gov/sports-betting-monthl...
7,DE,online_sports_betting,blocked,Official tables split casino sportsbooks vs Sp...,https://delottery.com/Sports-Lottery/Monthly-N...
8,FL,online_sports_betting,not_publicly_available,No official monthly online sportsbook series o...,https://flgaming.gov/pmw/statistics/
10,IL,online_sports_betting,partial,January 2020: No Sport Detail CSV for January ...,https://igb.illinois.gov/sports-wagering/sport...
11,IN,online_sports_betting,partial,2019-07-Revenue.xlsx: no online brand rows; 20...,https://www.in.gov/igc/publications/monthly-re...
12,KS,online_sports_betting,pdf_only_not_yet_parsed,Official monthly detail PDFs separate online/r...,https://www.kslottery.gov/publications/sports-...
13,KY,online_sports_betting,blocked,Current sports wagering market report is Table...,https://khrc.ky.gov/new_docs.aspx?cat=76&menui...
15,MA,online_sports_betting,pdf_only_not_yet_parsed,Official Category 1/3 sports wagering PDFs are...,https://massgaming.com/regulations/revenue/


## Row counts and period span

In [4]:
results = pd.read_csv(paths["gaming_results"])
print("gaming_results rows", len(results))
print(results.groupby(["state_code", "vertical"]).size().head(20))
results.groupby(["state_code", "vertical"]).agg(earliest=("period_start", "min"), latest=("period_end", "max"), n=("state_code", "size")).reset_index()

gaming_results rows 18103
state_code  vertical             
CT          online_casino             223
            online_sports_betting     174
DC          online_sports_betting      62
DE          online_casino             612
IA          online_sports_betting    1669
IL          online_sports_betting     891
IN          online_sports_betting    1025
LA          online_sports_betting      55
MD          online_sports_betting     351
MI          online_casino            1060
            online_sports_betting    1060
MO          online_sports_betting      72
NC          online_sports_betting      31
NH          online_sports_betting      80
NJ          online_casino             971
            online_sports_betting    1234
NY          online_sports_betting    2288
OH          online_sports_betting     745
PA          online_casino             936
            online_sports_betting    1060
dtype: int64


,state_code,vertical,earliest,latest,n
0,CT,online_casino,2021-10-01,2026-07-31,223
1,CT,online_sports_betting,2021-10-01,2026-07-31,174
2,DC,online_sports_betting,2024-07-01,2026-07-31,62
3,DE,online_casino,2013-11-01,2026-07-31,612
4,IA,online_sports_betting,2019-08-01,2026-07-31,1669
5,IL,online_sports_betting,2020-03-01,2026-06-30,891
6,IN,online_sports_betting,2019-10-01,2026-07-31,1025
7,LA,online_sports_betting,2021-07-01,2026-07-31,55
8,MD,online_sports_betting,2024-08-01,2026-07-31,351
9,MI,online_casino,2021-01-01,2026-07-31,1060


## State-period revenue by `revenue_basis`

In [5]:
state_period = pd.read_csv(paths["state_period_revenue"])
monthly = state_period[
    (state_period["frequency"] == "monthly")
    & (state_period["vertical"] == "online_sports_betting")
    & (state_period["channel"] == "online")
    & state_period["revenue"].notna()
].copy()
print("monthly online sports periods", len(monthly))
monthly.groupby("revenue_basis")["revenue"].sum()

monthly online sports periods 805


revenue_basis
AGR                5.706915e+09
GGR                8.072636e+20
net_proceeds       2.695845e+09
taxable_revenue    3.819409e+09
Name: revenue, dtype: float64

## Why bases must not be blindly summed

A naive total across every `revenue` value in one month mixes legal definitions.
Keep the label.

In [6]:
sample_month = "2025-12-01"
one = monthly[monthly["period_start"] == sample_month]
print("naive mixed-basis sum", float(one["revenue"].sum()) if len(one) else None)
one.groupby("revenue_basis")["revenue"].sum() if len(one) else one

naive mixed-basis sum 373053932396388.56


revenue_basis
AGR                1.568230e+08
GGR                3.730535e+14
net_proceeds       8.626253e+07
taxable_revenue    1.581268e+08
Name: revenue, dtype: float64

In [7]:
operators = pd.read_csv(paths["operator_revenue"])
print("operator rows", len(operators))
operators.head()

operator rows 16276


,jurisdiction,state_code,vertical,channel,operator,period_start,period_end,frequency,handle,gross_revenue,...,taxable_revenue,net_proceeds,tax,revenue,revenue_basis,reported_revenue_name,source_url,source_sha256,retrieved_at_utc,report_status
0,Connecticut,CT,online_casino,online,"MPI Master Wagering License CT, LLC",2021-10-01,2021-10-31,monthly,189562887.0,3610437.0,...,NaN,NaN,649879.0,3610437.0,GGR,Total Gross Gaming Revenue,https://data.ct.gov/api/views/imqd-at3c/rows.c...,e7b21691f7d424cf0bb624303d4cf6f23c62fadb320633...,2026-09-03T01:49:11.364254+00:00,ok
1,Connecticut,CT,online_casino,online,"Mohegan Digital, LLC",2021-10-01,2021-10-31,monthly,123299057.0,3030434.0,...,NaN,NaN,545478.0,3030434.0,GGR,Total Gross Gaming Revenue,https://data.ct.gov/api/views/imqd-at3c/rows.c...,e7b21691f7d424cf0bb624303d4cf6f23c62fadb320633...,2026-09-03T01:49:11.364254+00:00,ok
2,Connecticut,CT,online_casino,online,"MPI Master Wagering License CT, LLC",2021-11-01,2021-11-30,monthly,416047836.0,7931349.0,...,NaN,NaN,1427642.0,7931349.0,GGR,Total Gross Gaming Revenue,https://data.ct.gov/api/views/imqd-at3c/rows.c...,e7b21691f7d424cf0bb624303d4cf6f23c62fadb320633...,2026-09-03T01:49:11.364254+00:00,ok
3,Connecticut,CT,online_casino,online,"Mohegan Digital, LLC",2021-11-01,2021-11-30,monthly,275235600.0,5920099.0,...,NaN,NaN,1065618.0,5920099.0,GGR,Total Gross Gaming Revenue,https://data.ct.gov/api/views/imqd-at3c/rows.c...,e7b21691f7d424cf0bb624303d4cf6f23c62fadb320633...,2026-09-03T01:49:11.364254+00:00,ok
4,Connecticut,CT,online_casino,online,"MPI Master Wagering License CT, LLC",2021-12-01,2021-12-31,monthly,523770709.0,9199924.0,...,NaN,NaN,1655986.0,9199924.0,GGR,Total Gross Gaming Revenue,https://data.ct.gov/api/views/imqd-at3c/rows.c...,e7b21691f7d424cf0bb624303d4cf6f23c62fadb320633...,2026-09-03T01:49:11.364254+00:00,ok
